# 02. Дослідницький аналіз даних (EDA) та візуалізація

**Мета ноутбука:**
1. Швидко завантажити очищений набір даних (`4 618 780` рядків) із формату Apache Parquet.
2. Проаналізувати розподіл цільової змінної (`target`: позитивні проти негативних відгуків) та перевірити ступінь дисбалансу класів.
3. Дослідити топ-15 ігор за кількістю унікальних відгуків та їхньою часткою схвалення.
4. Перевірити залежність між довжиною відгуку (`word_count`), кількістю голосів корисності (`review_votes`) та його тональністю.
5. Зберегти 3 ключові графіки у високій роздільній здатності (`report/figures/`) для фінального звіту та презентації.

In [ ]:
import os
import time
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

PARQUET_PATH = "../data/steam_reviews_clean.parquet"
FIGURES_DIR = "../report/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# Налаштування стилю графіків
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 150

t0 = time.perf_counter()
df = pl.read_parquet(PARQUET_PATH)
print(f"Завантажено {df.height:,} рядків з Parquet за {time.perf_counter() - t0:.3f} с")

### 1. Баланс класів цільової змінної (`target`)

In [ ]:
class_dist = (
    df.group_by("target")
    .agg(pl.len().alias("count"))
    .with_columns((pl.col("count") / df.height * 100).round(2).alias("pct"))
    .sort("target", descending=True)
)
print(class_dist)

# Графік 1: Розподіл позитивних і негативних відгуків
dist_pd = class_dist.to_pandas()
dist_pd["label"] = dist_pd["target"].map({1: "Позитивний (1)", 0: "Негативний (0)"})

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(dist_pd["label"], dist_pd["count"] / 1e6, color=["#2ecc71", "#e74c3c"], width=0.5)
ax.set_title("Розподіл тональності відгуків у очищеному наборі Steam", fontsize=12, fontweight="bold")
ax.set_ylabel("Кількість відгуків (млн)")
ax.set_ylim(0, max(dist_pd["count"] / 1e6) * 1.18)

for bar, pct in zip(bars, dist_pd["pct"]):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.08, f"{h:.2f} млн\n({pct}%)", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/01_class_distribution.png", dpi=300)
plt.show()

### 2. Топ-15 ігор за кількістю відгуків та частка позитивних оцінок

In [ ]:
top_games = (
    df.group_by("app_name")
    .agg([
        pl.len().alias("total_reviews"),
        (pl.col("target").mean() * 100).round(1).alias("positive_rate")
    ])
    .filter(pl.col("app_name") != "Unknown Game")
    .sort("total_reviews", descending=True)
    .head(15)
)
print(top_games)

# Графік 2: Топ-15 ігор
top_pd = top_games.to_pandas().iloc[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top_pd["app_name"].astype(str), top_pd["total_reviews"] / 1e3, color="#3498db")
ax.set_title("Топ-15 ігор Steam за кількістю унікальних відгуків", fontsize=12, fontweight="bold")
ax.set_xlabel("Кількість відгуків (тис.)")
ax.set_xlim(0, max(top_pd["total_reviews"] / 1e3) * 1.22)

for bar, rate in zip(bars, top_pd["positive_rate"]):
    w = bar.get_width()
    ax.text(w + 1.5, bar.get_y() + bar.get_height()/2, f"{w:.1f}k ({rate}% +)", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/02_top_games.png", dpi=300)
plt.show()

### 3. Довжина відгуку (`word_count`) залежно від тональності

In [ ]:
length_stats = (
    df.group_by("target")
    .agg([
        pl.col("word_count").mean().round(1).alias("mean_words"),
        pl.col("word_count").median().alias("median_words"),
        pl.col("word_count").quantile(0.90).alias("p90_words"),
        pl.col("review_votes").mean().round(2).alias("mean_votes")
    ])
)
print(length_stats)

# Графік 3: Розподіл кількості слів (обрізаємо викиди > 200 слів для наочності)
sample_pd = (
    df.filter(pl.col("word_count") <= 200)
    .select(["target", "word_count"])
    .sample(n=300_000, seed=42)
    .to_pandas()
)
sample_pd["Тональність"] = sample_pd["target"].map({1: "Позитивний", 0: "Негативний"})

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(data=sample_pd, x="Тональність", y="word_count", palette={"Позитивний": "#2ecc71", "Негативний": "#e74c3c"}, ax=ax)
ax.set_title("Розподіл довжини відгуків (до 200 слів) за тональністю", fontsize=12, fontweight="bold")
ax.set_xlabel("")
ax.set_ylabel("Кількість слів у відгуку")

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/03_word_count_by_sentiment.png", dpi=300)
plt.show()